# ShowMe Core Methods

This notebook demonstrates the retrieval logic behind ShowMe without the Gradio UI and without processing real video files.

It covers:
- transcript / OCR chunk representation
- multilingual E5 embeddings
- Chroma vector search
- retrieval diagnostics
- conservative density boosting
- deterministic anchor selection
- the evidence block used for the RAG answer

The full app uses `intfloat/multilingual-e5-large`. This notebook defaults to the smaller E5 model so it runs comfortably in Colab.

In [ ]:
# Colab setup
!pip -q install sentence-transformers chromadb pandas numpy

In [ ]:
from dataclasses import dataclass
import math
import re
import uuid

import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

## 1. Toy Lecture Data

The real project stores four JSON files per video: `asr.json`, `ocr.json`, `clip.json`, and `meta.json`.

For this notebook we use a tiny corpus with ASR-like transcript chunks and OCR-like slide chunks. Notice that some text is English and some is Hebrew/transliterated Hebrew.

In [ ]:
toy_asr_segments = [
    {"video_id": "zvi_lecture_2026-04-26", "start": 120, "end": 150, "text": "Today we start with vector stores and semantic search."},
    {"video_id": "zvi_lecture_2026-04-26", "start": 170, "end": 210, "text": "An embedding is a numeric representation of text, useful for retrieval."},
    {"video_id": "zvi_lecture_2026-04-26", "start": 250, "end": 290, "text": "Now we compare a bi-encoder with a cross-encoder."},
    {"video_id": "zvi_lecture_2026-04-26", "start": 310, "end": 350, "text": "הבי אנקודר מחשב וקטור לשאילתה ולמסמך בנפרד."},
    {"video_id": "zvi_lecture_2026-04-26", "start": 2100, "end": 2140, "text": "This later section mentions embedding only briefly while discussing evaluation."},
    {"video_id": "lev_lecture_2026-03-18", "start": 600, "end": 640, "text": "Agents use tools and observations to decide the next action."},
]

toy_ocr_records = [
    {"video_id": "zvi_lecture_2026-04-26", "timestamp": 160, "text": "Embedding Strategies: semantic vectors, cosine similarity, retrieval"},
    {"video_id": "zvi_lecture_2026-04-26", "timestamp": 280, "text": "Bi-Encoder vs Cross-Encoder: speed vs accuracy"},
    {"video_id": "lev_lecture_2026-03-18", "timestamp": 610, "text": "Agent loop: plan, act, observe"},
]

pd.DataFrame(toy_asr_segments + toy_ocr_records)

## 2. Chunking

The app merges ASR segments so each vector represents a short thought rather than a tiny subtitle fragment. OCR records are already slide-sized.

In [ ]:
CHUNK_WINDOW_SEC = 60.0
CHUNK_MAX_CHARS = 1500

def chunk_asr(segments, window_sec=CHUNK_WINDOW_SEC):
    by_video = {}
    for seg in segments:
        by_video.setdefault(seg["video_id"], []).append(seg)

    chunks = []
    for video_id, segs in by_video.items():
        segs = sorted(segs, key=lambda x: x["start"])
        current = dict(segs[0])
        current["source"] = "asr"
        for seg in segs[1:]:
            span_after_merge = seg["end"] - current["start"]
            chars_after_merge = len(current["text"]) + 1 + len(seg["text"])
            if span_after_merge <= window_sec and chars_after_merge <= CHUNK_MAX_CHARS:
                current["end"] = seg["end"]
                current["text"] += " " + seg["text"]
            else:
                chunks.append(current)
                current = {**seg, "source": "asr"}
        chunks.append(current)
    return chunks

def chunk_ocr(records):
    return [
        {
            "video_id": r["video_id"],
            "start": r["timestamp"],
            "end": r["timestamp"],
            "text": r["text"],
            "source": "ocr",
        }
        for r in records
    ]

chunks = chunk_asr(toy_asr_segments) + chunk_ocr(toy_ocr_records)
pd.DataFrame(chunks)

## 3. Embedding + Chroma Index

E5 models use prefixes:
- documents: `passage: ...`
- queries: `query: ...`

The full app uses `intfloat/multilingual-e5-large`; here we use `intfloat/multilingual-e5-small` for speed.

In [ ]:
TEXT_MODEL_NAME = "intfloat/multilingual-e5-small"  # change to multilingual-e5-large to match the app more closely
model = SentenceTransformer(TEXT_MODEL_NAME)

client = chromadb.Client()
collection = client.create_collection(
    name=f"showme_core_demo_{uuid.uuid4().hex[:8]}",
    metadata={"hnsw:space": "cosine"},
)

ids = [f"{c['video_id']}__{i}" for i, c in enumerate(chunks)]
docs = [c["text"] for c in chunks]
metas = [{"video_id": c["video_id"], "start": c["start"], "end": c["end"], "source": c["source"]} for c in chunks]
embeddings = model.encode(["passage: " + d for d in docs], normalize_embeddings=True).tolist()

collection.add(ids=ids, documents=docs, metadatas=metas, embeddings=embeddings)
collection.count()

## 4. Retrieval + Diagnostics

This is the part that helps explain questions like: why does `encoder` retrieve Hebrew `אנקודר` even with no LLM?

Answer: the multilingual embedding model maps semantically related English/Hebrew/transliterated text into nearby vector space. No explicit dictionary is required.

In [ ]:
def hms(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:d}:{m:02d}:{s:02d}"

def exact_terms(query, text):
    found = []
    for term in re.findall(r"[\w'-]+", query, flags=re.UNICODE):
        if len(term) < 3:
            continue
        flags = re.IGNORECASE if term.isascii() else 0
        if re.search(re.escape(term), text, flags=flags):
            found.append(term)
    return sorted(set(found))

def search_text(query, n=8, source_filter=None):
    query_vec = model.encode(["query: " + query], normalize_embeddings=True)[0].tolist()
    where = {"source": source_filter} if source_filter else None
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=n,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    rows = []
    for rank, (doc, meta, dist) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), 1):
        score = 1 - dist / 2
        rows.append({
            "rank": rank,
            "vector_score": score,
            "video_id": meta["video_id"],
            "time": hms(meta["start"]),
            "source": meta["source"],
            "exact_terms": ", ".join(exact_terms(query, doc)) or "semantic-only",
            "text": doc,
        })
    return rows

pd.DataFrame(search_text("encoder"))

In [ ]:
# Try a few queries:
for q in ["embedding", "encoder", "אנקודר", "agent"]:
    print("\nQUERY:", q)
    display(pd.DataFrame(search_text(q, n=5))[ ["rank", "vector_score", "video_id", "time", "source", "exact_terms", "text"] ])

## 5. Optional Query Variants

The app can use an LLM to expand queries across English/Hebrew/transliteration. In deterministic mode, it does not.

Below we simulate expansion manually so the idea is visible without calling an external model.

In [ ]:
manual_variants = {
    "encoder": ["encoder", "אנקודר", "bi-encoder"],
    "embedding": ["embedding", "embeddings", "אמבדינג"],
}

def multi_query_search(query, n_per_variant=4):
    merged = {}
    for variant in manual_variants.get(query, [query]):
        for row in search_text(variant, n=n_per_variant):
            key = (row["video_id"], row["time"], row["source"])
            if key not in merged or row["vector_score"] > merged[key]["vector_score"]:
                row = dict(row)
                row["matched_variant"] = variant
                merged[key] = row
    return sorted(merged.values(), key=lambda r: -r["vector_score"])

pd.DataFrame(multi_query_search("encoder"))

## 6. Conservative Density Boost

Density boost rewards relevant hits that are supported by other nearby relevant hits in the same video. The important conservative rules are:

- weak hits cannot boost each other into the top result
- neighbors farther than 10 minutes do not count
- the total boost is capped
- we show whether support is literal (`exact=embedding`) or vector-only (`semantic-only`)

In [ ]:
DENSITY_WINDOW_SEC = 600.0
DENSITY_DECAY_SEC = 300.0
DENSITY_RELATIVE_FLOOR = 0.88
DENSITY_NEIGHBOR_WEIGHT = 0.35
DENSITY_MAX_BONUS = 0.65

def parse_hms_to_seconds(ts):
    parts = [int(p) for p in ts.split(":")]
    return parts[0] * 3600 + parts[1] * 60 + parts[2]

def density_boost(rows):
    if not rows:
        return []
    max_score = max(r["vector_score"] for r in rows)
    relevance_floor = max_score * DENSITY_RELATIVE_FLOOR
    denom = max(max_score - relevance_floor, 1e-9)
    boosted = []
    for r in rows:
        bonus = 0.0
        support = []
        if r["vector_score"] >= relevance_floor:
            for o in rows:
                if o is r or o["video_id"] != r["video_id"] or o["vector_score"] < relevance_floor:
                    continue
                dt = abs(parse_hms_to_seconds(o["time"]) - parse_hms_to_seconds(r["time"]))
                if dt > DENSITY_WINDOW_SEC:
                    continue
                neighbor_strength = (o["vector_score"] - relevance_floor) / denom
                source_diversity = 1.15 if o["source"] != r["source"] else 1.0
                bonus += neighbor_strength * source_diversity * math.exp(-dt / DENSITY_DECAY_SEC)
                support.append(f"{o['time']} {o['source']} exact={o['exact_terms']} vec={o['vector_score']:.3f}")
        bonus = min(DENSITY_MAX_BONUS, bonus * DENSITY_NEIGHBOR_WEIGHT)
        rr = dict(r)
        rr["density_bonus"] = bonus
        rr["final_score"] = r["vector_score"] * (1 + bonus)
        rr["density_support"] = "; ".join(support) or "-"
        boosted.append(rr)
    return sorted(boosted, key=lambda x: -x["final_score"])

rows = multi_query_search("embedding", n_per_variant=5)
pd.DataFrame(density_boost(rows))[ ["final_score", "vector_score", "density_bonus", "matched_variant", "video_id", "time", "source", "exact_terms", "density_support", "text"] ]

## 7. Anchor Selection

The app can ask an LLM to choose the best teaching moment from density-ranked candidates. For a transparent notebook demo, we use deterministic selection: top final score wins.

This is the same fallback used by the app when `LLM_PROVIDER=none`.

In [ ]:
ranked = density_boost(multi_query_search("embedding", n_per_variant=5))
anchor = ranked[0]
others = ranked[1:4]

print("ANCHOR")
print(anchor["video_id"], anchor["time"], anchor["source"])
print(anchor["text"])

pd.DataFrame([anchor] + others)[ ["final_score", "video_id", "time", "source", "matched_variant", "exact_terms", "text"] ]

## 8. RAG Evidence Prompt

The app's RAG answer does not retrieve external information. It builds an evidence block from the selected anchor plus nearby alternatives, then asks the selected LLM to answer only from those chunks.

This notebook prints the evidence block rather than calling an LLM.

In [ ]:
def rag_evidence_block(query, chunks):
    lines = []
    for c in chunks:
        lines.append(f"[{c['time']}] ({c['video_id']}, {c['source']}) {c['text'][:400]}")
    return f"Query: {query}\n\nAvailable lecture chunks:\n" + "\n".join(lines)

print(rag_evidence_block("embedding", [anchor] + others))

## What to Look For

- If `exact_terms` is `semantic-only`, the embedding model retrieved a semantically related chunk without a literal word match.
- If `density_bonus` is high but support is mostly `semantic-only`, density may be over-helping vector noise.
- If the top OCR slide is excellent but ASR is weird, the likely problem is speech transcript quality or chunking.
- If `encoder` retrieves `אנקודר` with no LLM, that is multilingual embedding behavior, not query expansion.